# **Semana 13: Simulación de Réplicas de Lectura con Hilos (NB2 - Práctico/Ejercicios)**

## **Propósito de la Sesión**
Simular el comportamiento de un sistema de bases de datos con **réplicas de lectura** utilizando hilos en Python. Aprenderemos a distribuir consultas entre múltiples réplicas, medir tiempos de respuesta y simular fallos para entender conceptos de alta disponibilidad y balanceo de carga.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Comprender** el concepto de réplicas de lectura en bases de datos.
2. **Simular** múltiples instancias de base de datos usando estructuras de datos en Python.
3. **Implementar** un balanceador de carga simple que distribuya consultas entre réplicas.
4. **Simular** fallos en réplicas y observar el impacto en la disponibilidad.
5. **Medir** tiempos de respuesta y throughput del sistema simulado.
6. **Reflexionar** sobre las estrategias de alta disponibilidad en sistemas reales.

## **Configuración Inicial**

Importamos las librerías necesarias para la simulación: `threading` para concurrencia, `time` para mediciones, `random` para simular variabilidad, y `queue` para gestionar peticiones.

In [ ]:
# Importaciones
import threading
import time
import random
import queue
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import copy

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

print("Librerías importadas correctamente.")
print(f"Número de hilos disponibles: {threading.active_count()}")

---
## **Parte 1: Conceptos de Réplicas de Lectura**

### **¿Qué son las réplicas de lectura?**

En sistemas de bases de datos, una **réplica de lectura** es una copia de la base de datos principal que se utiliza exclusivamente para consultas de lectura (SELECT). Las escrituras (INSERT, UPDATE, DELETE) van al nodo principal (maestro), y los cambios se replican a las réplicas.

### **Ventajas:**
*   **Escalabilidad**: Distribuye la carga de lecturas entre múltiples servidores.
*   **Disponibilidad**: Si una réplica falla, las otras siguen funcionando.
*   **Aislamiento**: Consultas pesadas (reportes) no afectan al sistema transaccional.

### **Desafíos:**
*   **Consistencia**: Las réplicas pueden estar ligeramente desactualizadas (replicación asíncrona).
*   **Balanceo de carga**: Necesitamos distribuir las consultas de manera equitativa.
*   **Detección de fallos**: El sistema debe detectar réplicas caídas y redirigir el tráfico.

En este notebook, simularemos estos conceptos usando hilos y estructuras de datos.

---
## **Parte 2: Diseño de la Simulación**

Simularemos:
*   **1 nodo maestro**: Maneja escrituras.
*   **N nodos réplica**: Manejan lecturas, con datos replicados desde el maestro.
*   **Balanceador**: Distribuye las consultas de lectura entre las réplicas.
*   **Clientes**: Generan peticiones de lectura y escritura concurrentemente.

### **Estructuras de datos:**
*   `datos_maestro`: Diccionario con los datos actuales (solo escrituras modifican aquí).
*   `replicas`: Lista de diccionarios que contienen copias de los datos (actualizados periódicamente).
*   `estado_replicas`: Lista de booleanos indicando si cada réplica está activa.
*   `cola_peticiones`: Cola de consultas a procesar.

In [ ]:
# Parámetros de la simulación
NUM_REPLICAS = 3
NUM_CLIENTES = 10
SIMULACION_SEGUNDOS = 15

# Datos iniciales (simulamos una tabla de productos)
datos_maestro = {
    1: {"nombre": "Laptop", "precio": 1200, "stock": 50},
    2: {"nombre": "Mouse", "precio": 25, "stock": 200},
    3: {"nombre": "Teclado", "precio": 80, "stock": 150},
    4: {"nombre": "Monitor", "precio": 300, "stock": 30},
    5: {"nombre": "Auriculares", "precio": 45, "stock": 100}
}

# Crear réplicas (copia inicial de los datos)
replicas = [copy.deepcopy(datos_maestro) for _ in range(NUM_REPLICAS)]
estado_replicas = [True] * NUM_REPLICAS  # Todas activas al inicio

# Cola de peticiones
cola_peticiones = queue.Queue()

# Estadísticas
stats_lecturas = []  # (timestamp, réplica_id, tiempo_respuesta)
stats_escrituras = []  # (timestamp, tiempo_respuesta)
lock_stats = threading.Lock()
lock_maestro = threading.Lock()
lock_replicas = [threading.Lock() for _ in range(NUM_REPLICAS)]

print(f"Sistema inicializado con {NUM_REPLICAS} réplicas.")
print(f"Datos maestro: {len(datos_maestro)} productos.")

---
## **Parte 3: Funciones del Sistema**

### **3.1. Balanceador de carga (round-robin simple)**

In [ ]:
class Balanceador:
    def __init__(self, num_replicas):
        self.num_replicas = num_replicas
        self.contador = 0
        self.lock = threading.Lock()
    
    def obtener_replica(self):
        """Retorna el índice de la siguiente réplica activa (round-robin)"""
        with self.lock:
            intentos = 0
            while intentos < self.num_replicas * 2:
                idx = self.contador % self.num_replicas
                self.contador += 1
                if estado_replicas[idx]:
                    return idx
                intentos += 1
            # Si todas están caídas, retornar None
            return None

balanceador = Balanceador(NUM_REPLICAS)

### **3.2. Procesador de lecturas**

In [ ]:
def procesar_lectura(cliente_id, consulta):
    """Simula una consulta de lectura en una réplica"""
    inicio = time.time()
    
    # Obtener réplica del balanceador
    replica_idx = balanceador.obtener_replica()
    
    if replica_idx is None:
        # No hay réplicas disponibles
        with lock_stats:
            stats_lecturas.append((time.time(), -1, -1, "fallo"))
        return None
    
    # Simular tiempo de consulta (entre 0.01 y 0.1 segundos)
    tiempo_consulta = random.uniform(0.01, 0.1)
    time.sleep(tiempo_consulta)
    
    # Realizar la consulta (solo lectura)
    with lock_replicas[replica_idx]:
        if consulta == "listar":
            resultado = list(replicas[replica_idx].values())
        elif consulta.startswith("obtener:"):
            producto_id = int(consulta.split(":")[1])
            resultado = replicas[replica_idx].get(producto_id, None)
        else:
            resultado = None
    
    fin = time.time()
    tiempo_respuesta = fin - inicio
    
    with lock_stats:
        stats_lecturas.append((time.time(), replica_idx, tiempo_respuesta, "exito"))
    
    return resultado

### **3.3. Procesador de escrituras**

In [ ]:
def procesar_escritura(cliente_id, operacion):
    """Simula una escritura en el maestro y posterior replicación"""
    inicio = time.time()
    
    # Simular tiempo de escritura
    tiempo_escritura = random.uniform(0.05, 0.2)
    time.sleep(tiempo_escritura)
    
    # Aplicar cambio en el maestro
    with lock_maestro:
        if operacion["tipo"] == "actualizar_precio":
            producto_id = operacion["producto_id"]
            nuevo_precio = operacion["nuevo_precio"]
            if producto_id in datos_maestro:
                datos_maestro[producto_id]["precio"] = nuevo_precio
        elif operacion["tipo"] == "reducir_stock":
            producto_id = operacion["producto_id"]
            cantidad = operacion["cantidad"]
            if producto_id in datos_maestro:
                datos_maestro[producto_id]["stock"] -= cantidad
    
    # Simular replicación a las réplicas (asíncrona, con retraso)
    def replicar():
        time.sleep(random.uniform(0.02, 0.1))  # Retraso de replicación
        with lock_maestro:
            datos_actualizados = copy.deepcopy(datos_maestro)
        for i in range(NUM_REPLICAS):
            if estado_replicas[i]:
                with lock_replicas[i]:
                    replicas[i] = copy.deepcopy(datos_actualizados)
    
    threading.Thread(target=replicar, daemon=True).start()
    
    fin = time.time()
    tiempo_respuesta = fin - inicio
    
    with lock_stats:
        stats_escrituras.append((time.time(), tiempo_respuesta))
    
    return True

### **3.4. Hilo cliente generador de peticiones**

In [ ]:
def cliente_worker(cliente_id):
    """Hilo que genera peticiones aleatorias"""
    fin_simulacion = time.time() + SIMULACION_SEGUNDOS
    
    while time.time() < fin_simulacion:
        # Elegir tipo de operación: 80% lecturas, 20% escrituras
        if random.random() < 0.8:
            # Lectura
            if random.random() < 0.5:
                consulta = "listar"
            else:
                producto_id = random.choice(list(datos_maestro.keys()))
                consulta = f"obtener:{producto_id}"
            resultado = procesar_lectura(cliente_id, consulta)
        else:
            # Escritura
            producto_id = random.choice(list(datos_maestro.keys()))
            if random.random() < 0.5:
                operacion = {
                    "tipo": "actualizar_precio",
                    "producto_id": producto_id,
                    "nuevo_precio": random.randint(10, 1500)
                }
            else:
                operacion = {
                    "tipo": "reducir_stock",
                    "producto_id": producto_id,
                    "cantidad": random.randint(1, 5)
                }
            procesar_escritura(cliente_id, operacion)
        
        # Pequeña pausa entre peticiones
        time.sleep(random.uniform(0.05, 0.3))

---
## **Parte 4: Simulación de Fallos en Réplicas**

Crearemos un hilo que simule fallos aleatorios en las réplicas (caídas y recuperaciones).

In [ ]:
def simulador_fallos():
    """Hilo que simula fallos y recuperaciones en las réplicas"""
    fin_simulacion = time.time() + SIMULACION_SEGUNDOS
    
    while time.time() < fin_simulacion:
        time.sleep(random.uniform(2, 5))  # Esperar entre fallos
        
        if random.random() < 0.7:  # 70% de probabilidad de fallo
            # Elegir réplica al azar
            replica_fallar = random.randint(0, NUM_REPLICAS - 1)
            if estado_replicas[replica_fallar]:
                estado_replicas[replica_fallar] = False
                print(f"\n🔴 RÉPLICA {replica_fallar} HA FALLADO a las {datetime.now().strftime('%H:%M:%S')}")
                
                # Programar recuperación después de un tiempo
                tiempo_recuperacion = random.uniform(3, 7)
                threading.Timer(tiempo_recuperacion, recuperar_replica, args=[replica_fallar]).start()

def recuperar_replica(replica_id):
    """Recupera una réplica y la sincroniza con el maestro"""
    if not estado_replicas[replica_id]:
        # Sincronizar datos con el maestro antes de activar
        with lock_maestro:
            with lock_replicas[replica_id]:
                replicas[replica_id] = copy.deepcopy(datos_maestro)
        estado_replicas[replica_id] = True
        print(f"\n🟢 RÉPLICA {replica_id} RECUPERADA a las {datetime.now().strftime('%H:%M:%S')}")

---
## **Parte 5: Ejecutar la Simulación**

In [ ]:
print("="*70)
print("INICIANDO SIMULACIÓN DE RÉPLICAS DE LECTURA")
print(f"Duración: {SIMULACION_SEGUNDOS} segundos")
print(f"Réplicas: {NUM_REPLICAS}, Clientes: {NUM_CLIENTES}")
print("="*70)

# Limpiar estadísticas
stats_lecturas.clear()
stats_escrituras.clear()

# Iniciar hilo de fallos
thread_fallos = threading.Thread(target=simulador_fallos, daemon=True)
thread_fallos.start()

# Iniciar clientes
clientes = []
for i in range(NUM_CLIENTES):
    t = threading.Thread(target=cliente_worker, args=(i,), daemon=True)
    t.start()
    clientes.append(t)

# Esperar a que termine la simulación
time.sleep(SIMULACION_SEGUNDOS)

print("\n" + "="*70)
print("SIMULACIÓN FINALIZADA")
print("="*70)

---
## **Parte 6: Análisis de Resultados**

### **6.1. Estadísticas generales**

In [ ]:
print("--- ESTADÍSTICAS GENERALES ---")

total_lecturas = len(stats_lecturas)
total_escrituras = len(stats_escrituras)
lecturas_exitosas = sum(1 for s in stats_lecturas if s[3] == "exito")
lecturas_fallidas = sum(1 for s in stats_lecturas if s[3] == "fallo")

print(f"Total lecturas: {total_lecturas}")
print(f"  - Exitosas: {lecturas_exitosas}")
print(f"  - Fallidas (sin réplica disponible): {lecturas_fallidas}")
print(f"Total escrituras: {total_escrituras}")

if lecturas_exitosas > 0:
    tiempos_lectura = [s[2] for s in stats_lecturas if s[3] == "exito"]
    print(f"\nTiempos de respuesta lecturas:")
    print(f"  - Promedio: {sum(tiempos_lectura)/len(tiempos_lectura):.4f}s")
    print(f"  - Mínimo: {min(tiempos_lectura):.4f}s")
    print(f"  - Máximo: {max(tiempos_lectura):.4f}s")

if total_escrituras > 0:
    tiempos_escritura = [s[1] for s in stats_escrituras]
    print(f"\nTiempos de respuesta escrituras:")
    print(f"  - Promedio: {sum(tiempos_escritura)/len(tiempos_escritura):.4f}s")
    print(f"  - Mínimo: {min(tiempos_escritura):.4f}s")
    print(f"  - Máximo: {max(tiempos_escritura):.4f}s")

### **6.2. Distribución de lecturas por réplica**

In [ ]:
# Conteo de lecturas por réplica
lecturas_por_replica = {i: 0 for i in range(NUM_REPLICAS)}
for s in stats_lecturas:
    if s[3] == "exito":
        lecturas_por_replica[s[1]] += 1

print("\nLecturas por réplica:")
for replica, count in lecturas_por_replica.items():
    estado = "🟢 ACTIVA" if estado_replicas[replica] else "🔴 INACTIVA"
    print(f"  Réplica {replica}: {count} lecturas ({estado})")

# Visualización
if sum(lecturas_por_replica.values()) > 0:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.bar(lecturas_por_replica.keys(), lecturas_por_replica.values(), color=['green' if estado_replicas[i] else 'red' for i in range(NUM_REPLICAS)])
    plt.xlabel('Réplica')
    plt.ylabel('Número de lecturas')
    plt.title('Distribución de Lecturas por Réplica')
    
    plt.subplot(1, 2, 2)
    tiempos_exito = [s[2] for s in stats_lecturas if s[3] == "exito"]
    tiempos_fallo = [s[2] for s in stats_lecturas if s[3] == "fallo" and s[2] != -1]
    plt.hist(tiempos_exito, bins=20, alpha=0.7, label='Exitosas')
    plt.xlabel('Tiempo de respuesta (s)')
    plt.ylabel('Frecuencia')
    plt.title('Distribución de Tiempos de Respuesta')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

### **6.3. Verificación de consistencia eventual**

Verificamos que después de la simulación, los datos en las réplicas coinciden con el maestro (aplicando el concepto de consistencia eventual).

In [ ]:
print("\n--- VERIFICACIÓN DE CONSISTENCIA ---")

with lock_maestro:
    datos_maestro_final = copy.deepcopy(datos_maestro)

consistentes = 0
for i in range(NUM_REPLICAS):
    with lock_replicas[i]:
        replica_data = replicas[i]
    if replica_data == datos_maestro_final:
        consistentes += 1
        print(f"✅ Réplica {i}: consistente")
    else:
        print(f"❌ Réplica {i}: NO consistente (puede deberse a fallos o replicación en curso)")

print(f"\n{consistentes} de {NUM_REPLICAS} réplicas consistentes con el maestro.")

# Mostrar diferencias si las hay
if consistentes < NUM_REPLICAS:
    print("\nNota: En sistemas reales con replicación asíncrona, las réplicas pueden estar ligeramente desactualizadas.")

---
## **Ejercicios Prácticos para el Estudiante**

Ahora modifica y extiende la simulación.

### **Ejercicio 1: Cambiar la estrategia de balanceo**

Modifica el balanceador para usar una estrategia diferente:
*   **Menor carga**: Seleccionar la réplica con menos lecturas en el último minuto.
*   **Aleatorio**: Seleccionar una réplica activa al azar.

Compara los resultados con la estrategia round-robin.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

class BalanceadorCarga:
    def __init__(self, num_replicas, estrategia="round_robin"):
        self.num_replicas = num_replicas
        self.estrategia = estrategia
        self.contador = 0
        self.lock = threading.Lock()
        # Para estrategia 'menor_carga', necesitas llevar conteo de lecturas recientes
    
    def obtener_replica(self):
        # Implementa aquí las diferentes estrategias
        pass

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Simular replicación síncrona**

Modifica el proceso de escritura para que sea **síncrono**: las escrituras no se confirman hasta que todas las réplicas activas hayan sido actualizadas. Mide cómo afecta esto al tiempo de respuesta de las escrituras.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

def procesar_escritura_sincrona(cliente_id, operacion):
    # Implementa escritura con replicación síncrona
    pass

# --- FIN DE TU CÓDIGO ---